# GeoStats output table semantics (Private Preview)

This read-only notebook explains how GeoStats shapes output tables. It uses illustrative tables rather than executable code, so it requires no Python environment or processing-engine package.

> **Important**
> GeoStats is available only to approved private-preview participants. API behavior and result schemas can change before general availability. Use the GeoStats OpenAPI specification as the authoritative API contract.

This notebook focuses on row cardinality, observation strategies, temporal periods, row statuses, coverage, counts, provenance, and estimates.

## Strategy and row cardinality

The observation strategy is the main control on row cardinality and raster overlap behavior.

| Mode | How observations are used | Result rows |
| --- | --- | --- |
| `item` | Each matching STAC item is processed independently. | One row for each feature-item pair whose geometry overlaps the item's raster grid. |
| `mosaic` | Overlapping observations are resolved into one image by an explicit priority. | One row per feature, or one row per feature and period. |
| `all` | Every valid pixel observation contributes to the requested statistics. | One row per feature, or one row per feature and period. |

Adding calculations or statistics adds columns, not rows. For example, calculation `ndvi` with statistics `mean` and `count` produces columns such as `ndvi_mean` and `ndvi_count`.

In `item` mode, a feature that overlaps no item produces no placeholder row. If no feature overlaps any item, a successful run can publish an empty table. An intersection with the rectangular raster grid produces a row even when the intersecting pixels are all nodata or removed by a quality mask.

## Item-mode overlap example

Each `row` cell below becomes one output row.

| Feature | item-A | item-B | item-C | Output rows |
| --- | --- | --- | --- | ---: |
| F1 | row | - | - | 1 |
| F2 | row | row | - | 2 |
| F3 | row | row | - | 2 |
| F4 | - | - | row: `noValidPixels` | 1 |
| F5 | - | - | - | 0 |

Although an estimate can report $5 \times 3 = 15$ candidate feature-item pairs, this example produces only 6 rows because most pairs don't overlap.

| geometry_id | item_id | ndvi_mean | ndvi_count | status |
| --- | --- | ---: | ---: | --- |
| F1 | item-A | 0.591 | 60.500 | `ok` |
| F2 | item-A | 0.540 | 76.545 | `ok` |
| F2 | item-B | 0.650 | 97.864 | `ok` |
| F3 | item-A | 0.448 | 15.600 | `ok` |
| F3 | item-B | 0.442 | 15.600 | `ok` |
| F4 | item-C | null | 0.000 | `noValidPixels` |

The values are illustrative. F2's rows summarize only the portion covered by each item. F3 represents the same ground once per acquisition. F4 intersects a raster grid but has no usable pixels. F5 is absent; anti-join output identifiers against input identifiers to find uncovered features.

## Mosaic and all modes

### `mosaic`

GeoStats orders scenes by `earliest`, `latest`, or `preferLeastCloudy`, then uses the first valid pixel in that order at each location. `preferLeastCloudy` ranks whole scenes by `eo:cloud_cover`; the quality mask still decides whether an individual pixel is valid. Statistics are calculated from the resolved mosaic.

### `all`

Every valid pixel observation contributes independently. Locations observed multiple times contribute multiple observations. An `all` result isn't generally the unweighted average of corresponding item-mode rows.

| Question | `mosaic` | `all` |
| --- | --- | --- |
| How many values can a location contribute? | At most one resolved pixel value. | One value from every valid observation. |
| Does scene priority matter? | Yes. | No mosaic priority is applied. |
| What does `<calculation>_count` represent? | Valid coverage-weighted pixel area. | Valid observation weight. |

## Temporal periods

For `mosaic` and `all`, an ISO 8601 duration such as `P1W` or `P1M` divides the requested `datetime` range into periods. Periods are anchored at the range start, not calendar boundaries. A final incomplete period is retained with `is_partial` set to `true`. Omitting the period treats the complete time range as one period.

A `P1W` period over `2026-06-03/2026-06-20` produces:

| period_start | period_end | is_partial |
| --- | --- | --- |
| 2026-06-03T00:00:00Z | 2026-06-10T00:00:00Z | false |
| 2026-06-10T00:00:00Z | 2026-06-17T00:00:00Z | false |
| 2026-06-17T00:00:00Z | 2026-06-20T00:00:00Z | true |

Each input feature receives one row for each period, including a period with no intersecting item or no usable pixel. The row status records that condition.

## Status and coverage

| Status | Meaning | Suggested action |
| --- | --- | --- |
| `ok` | The statistic was computed normally. | Use the value. |
| `belowValidFraction` | Some pixels were valid, but `spatial_valid_fraction` was below `minValidFraction`. | Relax the threshold, lengthen the period, or apply a downstream threshold. |
| `noValidPixels` | At least one scene intersected the feature, but every intersecting pixel was masked or nodata. | Lengthen the period or verify the quality mask. |
| `noItems` | No scene in the period intersected the feature. | Verify collection coverage, revisit interval, and time range. |

`minValidFraction` is evaluated per feature and period against `spatial_valid_fraction`, not against scene-level `eo:cloud_cover`. `item_count` includes intersecting scenes even when all their pixels are unusable, distinguishing `noValidPixels` from `noItems`. Run-level failures are reported on the asynchronous operation, not as result rows.

## Understanding `count`

With area weighting, boundary pixels contribute in proportion to polygon overlap, so a count can be fractional. Three fully covered valid pixels and one half-covered valid pixel produce $3 + 0.5 = 3.5$.

| Strategy | Interpretation of `<calculation>_count` |
| --- | --- |
| `item` | Valid coverage-weighted pixel area for that feature-item pair. |
| `mosaic` | Valid coverage-weighted pixel area in the resolved mosaic. |
| `all` | Valid observation weight across all contributing scenes. |

Because `all` counts repeated observations, its count and other statistics should not be interpreted as though each ground location contributed exactly once.

## Estimates, output, and provenance

The estimate endpoint plans geometry partitions and STAC searches without reading raster values or publishing a result. `FeatureItemPairs` is an upper bound based on feature and item search counts, not an expected result-row count. An estimate is also not a price or duration quote.

A successful run registers its result as a STAC item in the selected output collection. GeoParquet is the default file output; GeoJSON and Delta are also supported output choices. OGC API Features can expose enabled output collections as GeoJSON, and a collection-level SAS token provides temporary direct access to result assets.

Use a stable `geometryIdColumn` when submitting geometries. GeoStats carries that business identifier into the result so rows can be joined reliably to source data.

## Related documentation

- [GeoStats overview](./geostats-overview.md)
- [Use GeoStats to calculate zonal statistics](./use-geostats.md)
- [GeoStats OpenAPI specification](../spec/openapi.json)

The OpenAPI specification defines the request and response contract. The walkthrough covers authentication, formulas, computations, estimates, runs, operation monitoring, and result retrieval.